# Generar dataset simulado (CDMX) y unirlo con la base real — v2

Cambios respecto a la primera versión:
- Cada persona tiene **entre 6 y 10 transacciones** (antes eran siempre 4).
- Se agregó **mucha más variedad de comercios/descripciones** por categoría, para que el clasificador de gastos generalice mejor.
- Las personas simuladas se generan por "banda" objetivo (Sana / Observación / Riesgo) con ingreso, endeudamiento, ahorro y hábitos de gasto coherentes entre sí, para que el resultado final quede **balanceado en las 3 categorías** (antes "En observacion" quedaba casi vacía).
- El perfil financiero se sigue recalculando con la **regla oficial** (DTI + ahorro + comportamiento de gasto) para las 3000 personas, real y simulado por igual.

**Qué vas a hacer:** correr las celdas de arriba hacia abajo. Al final tendrás `dataset_financiero_completo.csv` **y** `dataset_financiero_completo.xlsx` en la misma carpeta.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

RUTA_BASE_REAL = "../data/raw/datos_financieros.csv"
TOTAL_PERSONAS_OBJETIVO = 3000

df_real = pd.read_csv(RUTA_BASE_REAL)
n_reales = df_real["id_usuario"].nunique()
siguiente_id = df_real["id_usuario"].max() + 1
n_simular = TOTAL_PERSONAS_OBJETIVO - n_reales
print(f"Reales: {n_reales}  Simular: {n_simular}")

## 1. Catálogos CDMX (variedad de comercios y descripciones)

In [ ]:
ESCOLARIDADES = ["Sin estudios", "Básica", "Media Superior", "Superior"]
PESOS_ESCOLARIDAD = [0.04, 0.38, 0.38, 0.20]

SERVICIOS_SALUD = ["IMSS", "ISSSTE", "Privado", "Ninguno", "IMSS-Bienestar"]
PESOS_SALUD = [0.30, 0.20, 0.22, 0.18, 0.10]

TIPOS_EMPLEO = ["Formal", "Informal", "Autoempleo"]
PESOS_EMPLEO = [0.50, 0.30, 0.20]

COMERCIOS_CDMX = {
    "Alimentacion": ["Superama", "Chedraui", "Central de Abastos", "Mercado sobre ruedas",
                      "OXXO", "7-Eleven", "Walmart Express", "Tortillería", "Panadería",
                      "La Comer", "Bodega Aurrerá", "Frutería del barrio"],
    "Vivienda": ["Pago de Renta", "Mantenimiento hogar", "Abono Hipoteca", "Cuota de condominio",
                 "Predial", "Reparación plomería"],
    "Servicios": ["Recibo CFE", "Pago de Agua (SACMEX)", "Internet Totalplay", "Recarga Telcel",
                  "Recarga AT&T", "Plan Movistar", "Gas natural"],
    "Transporte": ["Recarga tarjeta Metro/Metrobús", "Viaje DiDi", "Viaje Uber", "Gasolina Magna",
                   "Gasolina Premium", "Taxi de sitio", "Estacionamiento"],
    "Salud": ["Farmacia del Ahorro", "Farmacia Guadalajara", "Consulta médica",
              "Análisis Laboratorio", "Dentista", "Óptica"],
    "Ocio": ["Cine", "Suscripción Netflix", "Suscripción Spotify", "Salida fin de semana",
             "Restaurante", "Bar", "Concierto", "Boletos de evento"],
    "Otros": ["Ropa", "Tienda departamental", "Pago Elektra", "Abono Coppel",
              "Regalo cumpleaños", "Papelería", "Mascota - accesorios"],
}
CATEGORIAS = list(COMERCIOS_CDMX.keys())

## 2. Generar personas simuladas por "banda" objetivo

Cada persona se genera apuntando a una banda (Sana / Observación / Riesgo) que determina su rango de endeudamiento y su hábito de ahorro. Esto hace que, más adelante, la regla oficial de clasificación tenga señal clara para aprender — y que las 3 categorías queden balanceadas.

In [ ]:
BANDAS = np.random.choice(["sana", "obs", "riesgo"], size=n_simular, p=[0.34, 0.33, 0.33])

edades = np.random.randint(18, 76, size=n_simular)
sexos = np.random.choice(["M", "H"], size=n_simular)
escolaridades = np.random.choice(ESCOLARIDADES, size=n_simular, p=PESOS_ESCOLARIDAD)
servicios_salud = np.random.choice(SERVICIOS_SALUD, size=n_simular, p=PESOS_SALUD)
tipos_empleo = np.random.choice(TIPOS_EMPLEO, size=n_simular, p=PESOS_EMPLEO)
ingresos = np.round(np.clip(np.random.lognormal(mean=8.8, sigma=0.45, size=n_simular), 2800, 30000), 2)

dti = np.empty(n_simular)
ahorro = np.empty(n_simular, dtype=object)
for i, banda in enumerate(BANDAS):
    if banda == "sana":
        dti[i] = np.random.uniform(5, 29)
        ahorro[i] = np.random.choice(["Alta", "Media"], p=[0.6, 0.4])
    elif banda == "obs":
        dti[i] = np.random.uniform(30, 40)
        ahorro[i] = np.random.choice(["Media", "Baja"], p=[0.5, 0.5])
    else:
        dti[i] = np.random.uniform(41, 85)
        ahorro[i] = "Baja"
dti = np.round(dti).astype(int)

personas = pd.DataFrame({
    "id_usuario": np.arange(siguiente_id, siguiente_id + n_simular),
    "edad": edades, "sexo": sexos, "escolaridad": escolaridades,
    "servicio_salud": servicios_salud, "tipo_empleo": tipos_empleo,
    "ingreso_mensual": ingresos, "nivel_endeudamiento": dti, "frecuencia_ahorro": ahorro,
    "banda": BANDAS,
})
personas.head()

## 3. Generar entre 6 y 10 transacciones por persona

El % del ingreso que se gasta en cada categoría también depende de la banda (alguien "en riesgo" gasta proporcionalmente más en Vivienda + Alimentación, alguien "sano" gasta menos en Ocio/Otros), para que el comportamiento de gasto sea coherente con el perfil que le vamos a calcular después.

In [ ]:
PCT_POR_BANDA = {
    "sana":   {"Vivienda": 0.16, "Alimentacion": 0.028, "Servicios": 0.018, "Transporte": 0.018, "Salud": 0.012, "Ocio": 0.010, "Otros": 0.018},
    "obs":    {"Vivienda": 0.24, "Alimentacion": 0.038, "Servicios": 0.026, "Transporte": 0.028, "Salud": 0.018, "Ocio": 0.020, "Otros": 0.030},
    "riesgo": {"Vivienda": 0.32, "Alimentacion": 0.048, "Servicios": 0.028, "Transporte": 0.022, "Salud": 0.020, "Ocio": 0.022, "Otros": 0.045},
}
PESOS_CATEGORIA_EXTRA = {"Alimentacion": 0.34, "Transporte": 0.16, "Servicios": 0.12, "Salud": 0.10, "Ocio": 0.14, "Otros": 0.14}

filas = []
for _, p in personas.iterrows():
    ingreso = p["ingreso_mensual"]
    banda = p["banda"]
    pcts = PCT_POR_BANDA[banda]

    n_trans = np.random.randint(6, 11)  # 6 a 10 inclusive
    categorias_persona = ["Vivienda"]  # siempre hay renta/hipoteca
    restantes = n_trans - 1
    cats_extra = list(PESOS_CATEGORIA_EXTRA.keys())
    pesos_extra = list(PESOS_CATEGORIA_EXTRA.values())
    categorias_persona += list(np.random.choice(cats_extra, size=restantes, p=pesos_extra))

    for categoria in categorias_persona:
        descripcion = np.random.choice(COMERCIOS_CDMX[categoria])
        variacion = np.random.uniform(0.8, 1.2)
        valor = round(ingreso * pcts[categoria] * variacion, 2)
        filas.append({
            "id_usuario": p["id_usuario"], "edad": p["edad"], "sexo": p["sexo"],
            "escolaridad": p["escolaridad"], "servicio_salud": p["servicio_salud"],
            "tipo_empleo": p["tipo_empleo"], "ingreso_mensual": ingreso,
            "nivel_endeudamiento": p["nivel_endeudamiento"], "frecuencia_ahorro": p["frecuencia_ahorro"],
            "descripcion_transaccion": descripcion, "valor_transaccion": valor, "categoria_gasto": categoria,
        })

df_simulado = pd.DataFrame(filas)
columnas = ["id_usuario","edad","sexo","escolaridad","servicio_salud","tipo_empleo",
            "ingreso_mensual","nivel_endeudamiento","frecuencia_ahorro",
            "descripcion_transaccion","valor_transaccion","categoria_gasto"]
df_simulado = df_simulado[columnas]
print("Filas simuladas:", len(df_simulado), " Personas simuladas:", df_simulado.id_usuario.nunique())
print("Transacciones por persona (min/max):", df_simulado.groupby("id_usuario").size().min(), df_simulado.groupby("id_usuario").size().max())

## 4. Unir con la base real y calcular el perfil financiero con la regla oficial

In [ ]:
df_real_sin_perfil = df_real.drop(columns=["perfil_usuario"])
df_todas_trans = pd.concat([df_real_sin_perfil, df_simulado], ignore_index=True)

def calcular_variables_por_persona(df):
    base = df.groupby("id_usuario").agg(
        ingreso_mensual=("ingreso_mensual", "first"),
        nivel_endeudamiento=("nivel_endeudamiento", "first"),
        frecuencia_ahorro=("frecuencia_ahorro", "first"),
    )
    gasto_cat = df.pivot_table(index="id_usuario", columns="categoria_gasto",
                                values="valor_transaccion", aggfunc="sum", fill_value=0)
    def col(nombre):
        return gasto_cat[nombre] if nombre in gasto_cat.columns else 0
    gasto_total = gasto_cat.sum(axis=1)
    deuda_mensual = base["ingreso_mensual"] * base["nivel_endeudamiento"] / 100
    base["pct_gasto_total"] = gasto_total / base["ingreso_mensual"]
    base["pct_ocio_otros"] = (col("Ocio") + col("Otros")) / base["ingreso_mensual"]
    base["pct_estructural"] = (col("Vivienda") + col("Alimentacion") + deuda_mensual) / base["ingreso_mensual"]
    return base.reset_index()

vars_persona = calcular_variables_por_persona(df_todas_trans)

def clasificar_perfil_financiero(fila):
    d = fila["nivel_endeudamiento"]; a = fila["frecuencia_ahorro"]
    p_sana = int(d < 30) + int(a in ("Alta","Media")) + int(fila["pct_ocio_otros"] <= 0.20)
    p_obs = int(30 <= d <= 40) + int(a in ("Media","Baja")) + int(fila["pct_gasto_total"] >= 0.85)
    p_riesgo = int(d > 40) + int(a == "Baja") + int(fila["pct_estructural"] > 0.70)
    puntajes = {"En riesgo": p_riesgo, "En observacion": p_obs, "Finanzas sanas": p_sana}
    m = max(puntajes.values())
    for c in ["En riesgo", "En observacion", "Finanzas sanas"]:
        if puntajes[c] == m:
            return c

vars_persona["perfil_usuario"] = vars_persona.apply(clasificar_perfil_financiero, axis=1)
print("Distribución final de perfiles:")
print(vars_persona["perfil_usuario"].value_counts())
print(vars_persona["perfil_usuario"].value_counts(normalize=True).round(3))

## 5. Guardar el archivo final (CSV y Excel)

In [ ]:
df_final = df_todas_trans.merge(vars_persona[["id_usuario","perfil_usuario"]], on="id_usuario", how="left")

import unicodedata
def quitar_acentos(texto):
    if not isinstance(texto, str):
        return texto
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

# El equipo pidio que todo el dataset vaya sin acentos (compatibilidad con el backend)
for columna in df_final.select_dtypes(include="object").columns:
    df_final[columna] = df_final[columna].apply(quitar_acentos)

print("Total filas:", len(df_final), " Total personas:", df_final.id_usuario.nunique())

df_final.to_csv("../data/processed/dataset_financiero_completo.csv", index=False)
df_final.to_excel("../data/processed/dataset_financiero_completo.xlsx", index=False, engine="openpyxl")
print("Archivos guardados en: ../data/processed/dataset_financiero_completo.csv y .xlsx")

## Listo

Tienes `dataset_financiero_completo.csv` y `dataset_financiero_completo.xlsx` con 3000 personas (550 reales sin modificar + 2450 simuladas de CDMX), entre 6 y 10 transacciones cada una, y perfil financiero balanceado según la regla oficial del equipo.

**Siguiente paso**: en `Notebook_Entrenamiento.ipynb`, sección 1, pon:
```python
RUTA_DATASET = "dataset_financiero_completo.csv"
```
y corre todo el notebook.